<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row represents one pseudonymized content page at a March 2026 decision point.** The page-level row is built from the daily performance history available up to the end of March 2026.

For the feature window, I use the **90 days before and including 2026-03-31**. This gives a consistent historical window for measuring search visibility, clicks, position, and content lifecycle signals before making a ranking decision.

My lane is **Refresh / Content Opportunity Scoring**. The practical decision is which pages should be reviewed first for refresh or related content action. I use observed negative movement as a **proxy/evaluation signal**, not as proof that a refresh will cause recovery.

The warehouse source is `fact_content_daily_performance`, with `dim_content` used only when content-level metadata is needed. The warehouse daily table has the grain of one report date × client × content item, so I aggregate it to my page-level decision grain rather than treating each daily row as an independent page.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
I use five features for the initial ranking frame:
1. **`content_age_days`** — age of the content at the decision point.
2. **`days_since_last_update`** — freshness/lifecycle signal available before the decision.
3. **`impressions`** — search visibility/exposure observed during the feature window.
4. **`avg_position`** — average search position observed during the feature window.
5. **`ctr`** — click-through rate calculated from observed impressions and clicks during the feature window.

### Label / proxy
The evaluation proxy is **negative movement**, represented by the warehouse's observed trend signal. This is a proxy for pages that may deserve review; it is not a causal label saying that a refresh is required or guaranteed to work.

### Context
The following fields provide context rather than model features:
* `client_hash_id`
* `content_hash_id`
* `report_date`
* data-availability indicators
* content metadata used only to understand the slice

### Excluded
I deliberately exclude **`trend_pct` and `trend_direction` from the honest feature set** because they describe movement that is also used to define the outcome/proxy. Including them would allow information about the answer to enter the features and would create leakage.
I also exclude any product decision fields such as `health_score`, `priority_score`, `action_type`, or refresh flags. These are product decisions rather than independent observable signals and should not become model features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

if HF_TOKEN:
    print("HF_TOKEN loaded successfully.")
else:
    print("HF_TOKEN not found.")

HF_TOKEN loaded successfully.


In [11]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [12]:
FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
    LIMIT 5
""").df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ["HF_TOKEN"]

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

BASE = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"

KeyError: 'HF_TOKEN'

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.